In [ ]:
! pip install pandas numpy scikit-learn lightgbm xgboost catboost optuna matplotlib seaborn

In [ ]:
# =============================================================================
# CRAB AGE PREDICTION
# Based on previous experiments: rounding helps, no transform needed, skip stacking
# =============================================================================

import pandas as pd
import numpy as np
import optuna
import json
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_FOLDS = 5
N_TRIALS = 30
SEEDS = [42, 99, 2024]  # 3 seeds for averaging

# Set to False if you have the best parameters, True to tune the best parameters
RUN_TUNING = True
PARAMS_FILE = "best_params.json"

# =============================================================================
# STEP 1: DATA + FEATURES
# =============================================================================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

def add_features(df):
    df = df.copy()
    df["Sex_F"] = (df["Sex"] == "F").astype(int)
    df["Sex_M"] = (df["Sex"] == "M").astype(int)
    df["Sex_I"] = (df["Sex"] == "I").astype(int)
    df["Volume"] = df["Length"] * df["Diameter"] * df["Height"]
    df["Density"] = df["Weight"] / (df["Volume"] + 1e-6)
    df["Shucked_ratio"] = df["Shucked Weight"] / (df["Weight"] + 1e-6)
    df["Shell_ratio"] = df["Shell Weight"] / (df["Weight"] + 1e-6)
    df["Viscera_ratio"] = df["Viscera Weight"] / (df["Weight"] + 1e-6)
    df["Shell_to_Shucked"] = df["Shell Weight"] / (df["Shucked Weight"] + 1e-6)
    df["BMI"] = df["Weight"] / (df["Length"] ** 2 + 1e-6)
    df["Weight_remainder"] = (df["Weight"] - df["Shucked Weight"]
                              - df["Viscera Weight"] - df["Shell Weight"])
    df["Weight_cbrt"] = np.cbrt(df["Weight"])
    df["log_Shell"] = np.log1p(df["Shell Weight"])
    return df

train = add_features(train)
test = add_features(test)
feature_cols = [c for c in train.columns if c not in ["id", "Age", "Sex"]]
X = train[feature_cols].values
y = train["Age"].values
X_test = test[feature_cols].values
print(f"Features: {len(feature_cols)}")

# =============================================================================
# STEP 2: TUNING (or load saved params)
# =============================================================================
if RUN_TUNING:
    print("\n--- Optuna Tuning ---")
    folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X))

    def tune_lgb(trial):
        p = {"objective": "mae", "verbosity": -1, "n_jobs": -1,
             "random_state": SEED, "n_estimators": 1500,
             "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
             "max_depth": trial.suggest_int("max_depth", 4, 9),
             "num_leaves": trial.suggest_int("num_leaves", 20, 120),
             "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
             "subsample": trial.suggest_float("subsample", 0.6, 1.0),
             "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
             "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 3.0, log=True),
             "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 3.0, log=True)}
        scores = []
        for tr, val in folds:
            m = lgb.LGBMRegressor(**p)
            m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
            scores.append(mean_absolute_error(y[val], m.predict(X[val])))
        return np.mean(scores)

    def tune_xgb(trial):
        p = {"objective": "reg:absoluteerror", "verbosity": 0, "n_jobs": -1,
             "random_state": SEED, "n_estimators": 1500,
             "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
             "max_depth": trial.suggest_int("max_depth", 4, 9),
             "min_child_weight": trial.suggest_int("min_child_weight", 5, 40),
             "subsample": trial.suggest_float("subsample", 0.6, 1.0),
             "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
             "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 3.0, log=True),
             "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 3.0, log=True)}
        scores = []
        for tr, val in folds:
            m = xgb.XGBRegressor(**p)
            m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])], verbose=False)
            scores.append(mean_absolute_error(y[val], m.predict(X[val])))
        return np.mean(scores)

    def tune_cb(trial):
        p = {"loss_function": "MAE", "verbose": 0, "random_seed": SEED,
             "iterations": 1500,
             "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
             "depth": trial.suggest_int("depth", 4, 9),
             "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 8.0, log=True),
             "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0)}
        scores = []
        for tr, val in folds:
            m = cb.CatBoostRegressor(**p)
            m.fit(X[tr], y[tr], eval_set=(X[val], y[val]),
                  early_stopping_rounds=50, verbose=0)
            scores.append(mean_absolute_error(y[val], m.predict(X[val])))
        return np.mean(scores)

    # Run tuning
    s1 = optuna.create_study(direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED))
    s1.optimize(tune_lgb, n_trials=N_TRIALS, show_progress_bar=True)
    print(f"  LGB: {s1.best_value:.4f}")

    s2 = optuna.create_study(direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED))
    s2.optimize(tune_xgb, n_trials=N_TRIALS, show_progress_bar=True)
    print(f"  XGB: {s2.best_value:.4f}")

    s3 = optuna.create_study(direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED))
    s3.optimize(tune_cb, n_trials=N_TRIALS, show_progress_bar=True)
    print(f"  CB:  {s3.best_value:.4f}")

    # Extract and save best params
    tuned = {
        "lgb": s1.best_params,
        "xgb": s2.best_params,
        "cb":  s3.best_params,
        "scores": {
            "lgb": s1.best_value,
            "xgb": s2.best_value,
            "cb":  s3.best_value,
        }
    }
    with open(PARAMS_FILE, "w") as f:
        json.dump(tuned, f, indent=2)
    print(f"\n  Params saved to {PARAMS_FILE}")
    print(json.dumps(tuned, indent=2))

else:
    # Load previously saved params
    with open(PARAMS_FILE, "r") as f:
        tuned = json.load(f)
    print(f"Loaded params from {PARAMS_FILE}")
    print(json.dumps(tuned, indent=2))

# Build full param dicts from tuned values
lgb_p = {
    "objective": "mae", "verbosity": -1, "n_jobs": -1,
    "n_estimators": 1500, **tuned["lgb"]
}
xgb_p = {
    "objective": "reg:absoluteerror", "verbosity": 0, "n_jobs": -1,
    "n_estimators": 1500, **tuned["xgb"]
}
cb_p = {
    "loss_function": "MAE", "verbose": 0,
    "iterations": 1500, **tuned["cb"]
}

# =============================================================================
# STEP 3: MULTI-SEED TRAINING
# =============================================================================
print(f"\n--- Training ({len(SEEDS)} seeds × 3 models × {N_FOLDS} folds) ---")

all_oof, all_test = {}, {}

for seed in SEEDS:
    sf = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(X))

    for name, params in [("lgb", lgb_p), ("xgb", xgb_p), ("cb", cb_p)]:
        oof = np.zeros(len(X))
        tp = np.zeros(len(X_test))
        p = params.copy()

        # Set seed per model
        if name == "cb":
            p["random_seed"] = seed
        else:
            p["random_state"] = seed

        for tr, val in sf:
            if name == "lgb":
                m = lgb.LGBMRegressor(**p)
                m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])],
                      callbacks=[lgb.early_stopping(80, verbose=False),
                                 lgb.log_evaluation(0)])
            elif name == "xgb":
                m = xgb.XGBRegressor(**p)
                m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])], verbose=False)
            else:
                m = cb.CatBoostRegressor(**p)
                m.fit(X[tr], y[tr], eval_set=(X[val], y[val]),
                      early_stopping_rounds=80, verbose=0)

            oof[val] = m.predict(X[val])
            tp += m.predict(X_test) / N_FOLDS

        mae = mean_absolute_error(y, oof)
        print(f"  {name.upper()} seed={seed}: {mae:.4f}")
        all_oof[(name, seed)] = oof
        all_test[(name, seed)] = tp

# =============================================================================
# STEP 4: FIND BEST WEIGHTS + ROUNDING
# =============================================================================
print("\n--- Finding Best Approach ---")

# Average each model across seeds
lgb_oof = np.mean([all_oof[("lgb", s)] for s in SEEDS], axis=0)
xgb_oof = np.mean([all_oof[("xgb", s)] for s in SEEDS], axis=0)
cb_oof  = np.mean([all_oof[("cb", s)]  for s in SEEDS], axis=0)
lgb_test = np.mean([all_test[("lgb", s)] for s in SEEDS], axis=0)
xgb_test = np.mean([all_test[("xgb", s)] for s in SEEDS], axis=0)
cb_test  = np.mean([all_test[("cb", s)]  for s in SEEDS], axis=0)

# Search weights with rounding baked in
best_w_mae = float("inf")
best_w = (1/3, 1/3, 1/3)

for w1 in np.arange(0.1, 0.8, 0.05):
    for w2 in np.arange(0.05, 0.8, 0.05):
        w3 = round(1.0 - w1 - w2, 2)
        if w3 < 0.05:
            continue
        blend = w1 * lgb_oof + w2 * xgb_oof + w3 * cb_oof
        mae = mean_absolute_error(y, np.round(blend))
        if mae < best_w_mae:
            best_w_mae = mae
            best_w = (round(w1, 2), round(w2, 2), round(w3, 2))

# All candidates
candidates = {
    "LGB only (rounded)": (
        mean_absolute_error(y, np.round(lgb_oof)),
        np.round(lgb_test)
    ),
    "Equal avg (rounded)": (
        mean_absolute_error(y, np.round((lgb_oof + xgb_oof + cb_oof) / 3)),
        np.round((lgb_test + xgb_test + cb_test) / 3)
    ),
    f"Weighted {best_w} (rounded)": (
        best_w_mae,
        np.round(best_w[0] * lgb_test + best_w[1] * xgb_test + best_w[2] * cb_test)
    ),
}

# Print comparison and pick winner
best_name, best_mae, best_preds = None, float("inf"), None
for name, (mae, preds) in candidates.items():
    print(f"  {name:<40s}: MAE = {mae:.4f}")
    if mae < best_mae:
        best_name, best_mae, best_preds = name, mae, preds

print(f"\n  BEST: {best_name} → MAE = {best_mae:.4f}")

Features: 20

--- Optuna Tuning ---


  0%|          | 0/30 [00:00<?, ?it/s]

  LGB: 1.2866


  0%|          | 0/30 [00:00<?, ?it/s]

  XGB: 1.2899


  0%|          | 0/30 [00:00<?, ?it/s]

  CB:  1.2888

  Params saved to best_params.json
{
  "lgb": {
    "learning_rate": 0.022472426780406705,
    "max_depth": 9,
    "num_leaves": 117,
    "min_child_samples": 51,
    "subsample": 0.7218455076693483,
    "colsample_bytree": 0.5488360570031919,
    "reg_alpha": 0.2394204262539026,
    "reg_lambda": 0.03392060513051208
  },
  "xgb": {
    "learning_rate": 0.0328364298843639,
    "max_depth": 4,
    "min_child_weight": 22,
    "subsample": 0.9003988944050104,
    "colsample_bytree": 0.5004845791155039,
    "reg_alpha": 0.2877404726438605,
    "reg_lambda": 0.01762651588585434
  },
  "cb": {
    "learning_rate": 0.03844365649887091,
    "depth": 8,
    "l2_leaf_reg": 3.36067357285491,
    "bagging_temperature": 0.5444748511100536
  },
  "scores": {
    "lgb": 1.2866042515404372,
    "xgb": 1.289875872898102,
    "cb": 1.28876815531989
  }
}

--- Training (3 seeds × 3 models × 5 folds) ---
  LGB seed=42: 1.2865
  XGB seed=42: 1.2899
  CB seed=42: 1.2888
  LGB seed=99: 1.2897


# Results:
## Features

**20**

---

## Optuna Tuning

### LGB

* **Best trial:** 4
* **Best value:** 1.2866
* **Progress:** `30/30 [03:02<00:00, 6.25s/it]`

### XGB

* **Best trial:** 18
* **Best value:** 1.28988
* **Progress:** `30/30 [23:31<00:00, 44.10s/it]`

### CB

* **Best trial:** 28
* **Best value:** 1.28877
* **Progress:** `30/30 [12:11<00:00, 30.43s/it]`

### Summary Table

| Model | Best Trial | Best Value | Progress                         |
| ----- | ---------: | ---------: | -------------------------------- |
| LGB   |          4 |     1.2866 | `30/30 [03:02<00:00, 6.25s/it]`  |
| XGB   |         18 |    1.28988 | `30/30 [23:31<00:00, 44.10s/it]` |
| CB    |         28 |    1.28877 | `30/30 [12:11<00:00, 30.43s/it]` |

**Params saved to:** `best_params.json`

```json
{
  "lgb": {
    "learning_rate": 0.022472426780406705,
    "max_depth": 9,
    "num_leaves": 117,
    "min_child_samples": 51,
    "subsample": 0.7218455076693483,
    "colsample_bytree": 0.5488360570031919,
    "reg_alpha": 0.2394204262539026,
    "reg_lambda": 0.03392060513051208
  },
  "xgb": {
    "learning_rate": 0.0328364298843639,
    "max_depth": 4,
    "min_child_weight": 22,
    "subsample": 0.9003988944050104,
    "colsample_bytree": 0.5004845791155039,
    "reg_alpha": 0.2877404726438605,
    "reg_lambda": 0.01762651588585434
  },
  "cb": {
    "learning_rate": 0.03844365649887091,
    "depth": 8,
    "l2_leaf_reg": 3.36067357285491,
    "bagging_temperature": 0.5444748511100536
  },
  "scores": {
    "lgb": 1.2866042515404372,
    "xgb": 1.289875872898102,
    "cb": 1.28876815531989
  }
}
```

---

## Training (3 seeds × 3 models × 5 folds)

| Model | Seed |  Score |
| ----- | ---: | -----: |
| LGB   |   42 | 1.2865 |
| XGB   |   42 | 1.2899 |
| CB    |   42 | 1.2888 |
| LGB   |   99 | 1.2897 |
| XGB   |   99 | 1.2926 |
| CB    |   99 | 1.2882 |
| LGB   | 2024 | 1.2856 |
| XGB   | 2024 | 1.2912 |
| CB    | 2024 | 1.2894 |

---

## Finding Best Approach

| Method                                                                         |    MAE |
| ------------------------------------------------------------------------------ | -----: |
| LGB only (rounded)                                                             | 1.2601 |
| Equal avg (rounded)                                                            | 1.2573 |
| Weighted (`np.float64(0.45)`, `np.float64(0.2)`, `np.float64(0.35)`) (rounded) | 1.2559 |

### Best Result

**BEST:** Weighted (`np.float64(0.45)`, `np.float64(0.2)`, `np.float64(0.35)`) (rounded) → **MAE = 1.2559**


In [ ]:
# =============================================================================
# STEP 5: GENERATE SUBMISSION
# =============================================================================
print("\n--- Submission ---")

# Use the best approach
final_test = best_w[0] * lgb_test + best_w[1] * xgb_test + best_w[2] * cb_test
final_preds = np.clip(np.round(final_test), 1, None)

submission = pd.DataFrame({"id": test["id"], "Age": final_preds.astype(int)})
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv ({len(submission)} rows)")
print(f"Prediction range: {final_preds.min():.0f} - {final_preds.max():.0f}")
print(submission.head(10))


--- Submission ---
Saved submission.csv (10000 rows)
Prediction range: 4 - 18
      id  Age
0  15000   12
1  15001   10
2  15002    6
3  15003   11
4  15004    9
5  15005    7
6  15006    6
7  15007    5
8  15008   13
9  15009   14
